# 05 · Visitantes recurrentes: DBSCAN sobre desconocidos (no supervisado, I3 e I4)

Se ejecuta en **Google Colab** (la GPU solo acelera sacar los embeddings de LFW), de arriba abajo.

**Qué se mide.** Los desconocidos de la portería no tienen etiqueta, así que DBSCAN (distancia coseno) los agrupa y la regla I4
alerta si un clúster tiene ≥ 3 apariciones en ≥ 3 días distintos. Para saber si las alertas aciertan se **simula** un escenario con
fotos de LFW donde se conoce quién es quién: recurrentes (deben alertar), gente que viene varias veces el **mismo día** o solo
**dos días** (no deben alertar) y apariciones sueltas (ruido).

**Cómo se evita el autoengaño.** `eps` se elige con un grupo de personas (ajuste) y el resultado se mide con **otras personas
que nunca se usaron** (prueba), repetido con 5 escenarios distintos. Es una simulación: declararlo así en el informe.

In [ ]:
import os, zipfile
from google.colab import files

subidos = files.upload()          # selecciona solo ml.zip (carpeta ml/ del repo, sin data/)
nombre = list(subidos)[0]         # Colab renombra a "ml (1).zip" si ya existía: usa el nombre real
os.makedirs("/content/condohub-ia", exist_ok=True)
zipfile.ZipFile(nombre).extractall("/content/condohub-ia")
os.environ["ML_DATA_DIR"] = "/content/condohub-ml"
os.chdir("/content/condohub-ia")
print(os.listdir("/content/condohub-ia"))

In [ ]:
!pip install -q facenet-pytorch --no-deps
!pip install -q scikit-learn pyyaml matplotlib pillow
import torch
print("GPU disponible:", torch.cuda.is_available())

## 1 · Embeddings de LFW (personas reales, con varias fotos cada una)

In [ ]:
import json, yaml, numpy as np
import matplotlib.pyplot as plt
from ml.src import embeddings, lfw, cluster, evaluacion_cluster as ev, escenario_desconocidos as esc
from ml.src.split import dividir_por_persona
from ml.src.paths import dir_experimento

cfg = yaml.safe_load(open("ml/configs/dbscan_desconocidos.yaml", encoding="utf-8"))
imagenes, etiquetas, _ = lfw.cargar_lfw(cfg["lfw_min_fotos"])
embeber = embeddings.crear_embebedor()
emb, detectado = embeber(imagenes)
emb, etiquetas = emb[detectado], etiquetas[detectado]
del imagenes
print(f"{len(emb)} fotos de {len(np.unique(etiquetas))} personas con rostro detectado")

In [ ]:
partes = dividir_por_persona(etiquetas, tuple(cfg["proporciones"]), cfg["semilla"])
personas_ajuste = np.unique(etiquetas[partes["train"]])
personas_prueba = np.unique(etiquetas[partes["test"]])
assert not set(personas_ajuste) & set(personas_prueba)
print("personas para elegir eps:", len(personas_ajuste), "| personas para medir:", len(personas_prueba))

## 2 · Elegir `eps`

Primero la guía del plan: el **codo del gráfico k-distance** (k = `min_samples`). Después se barre una rejilla de `eps`
y se mira cuántas alertas aciertan (precisión) y cuántos recurrentes se detectan (recall), siempre con las personas de ajuste.

In [ ]:
e_aj = esc.construir(emb, etiquetas, personas_ajuste, **cfg["escenario_ajuste"],
                     ventana_dias=cfg["ventana_dias"], semilla=cfg["semilla"])
print("escenario de ajuste:", len(e_aj.personas), "apariciones de", len(set(e_aj.personas.tolist())), "personas;",
      "recurrentes reales:", len(ev.personas_recurrentes(e_aj.personas, e_aj.dias, cfg["min_apariciones"], cfg["min_dias"])))

dk = cluster.distancias_k(e_aj.embeddings, k=cfg["min_samples"])
eps_codo = ev.codo(dk)
print("eps sugerido por el codo del k-distance:", round(eps_codo, 3))

destino = dir_experimento(cfg["nombre"])
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(dk); ax.axhline(eps_codo, color="red", ls="--", label=f"codo = {eps_codo:.2f}")
ax.set_xlabel("apariciones ordenadas"); ax.set_ylabel(f"distancia coseno al vecino {cfg['min_samples']}")
ax.set_title("k-distance"); ax.legend(); fig.tight_layout()
fig.savefig(destino / "k_distance.png", dpi=150); plt.show()

In [ ]:
filas = ev.barrer_eps(e_aj.embeddings, e_aj.personas, e_aj.dias, cfg["eps_grid"], cfg["min_samples"],
                      cfg["min_apariciones"], cfg["min_dias"])
print(f"{'eps':>5} {'clústeres':>9} {'ruido':>5} {'silh.':>6} {'alertas':>7} {'precisión':>9} {'recall':>6} {'F1':>5}")
for f in filas:
    sil = "  -  " if f["silhouette"] is None else f"{f['silhouette']:.2f}"
    pre = "  -  " if f["precision"] is None else f"{f['precision']:.2f}"
    print(f"{f['eps']:>5.2f} {f['clusters']:>9} {f['ruido']:>5} {sil:>6} {f['alertas']:>7} {pre:>9} {f['recall']:>6.2f} {ev.f1(f):>5.2f}")

# Se elige el eps de mejor F1; a igualdad, el más cercano al codo
eps_elegido = min(filas, key=lambda f: (-ev.f1(f), abs(f["eps"] - eps_codo)))["eps"]
print("\neps elegido (mejor F1 en el ajuste):", eps_elegido)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot([f["eps"] for f in filas], [f["precision"] or 0 for f in filas], "o-", label="precisión de las alertas")
ax.plot([f["eps"] for f in filas], [f["recall"] for f in filas], "s-", label="recall de recurrentes")
ax.axvline(eps_elegido, color="green", ls="--", label=f"elegido = {eps_elegido}")
ax.axvline(eps_codo, color="red", ls=":", label=f"codo = {eps_codo:.2f}")
ax.set_xlabel("eps"); ax.legend(); fig.tight_layout()
fig.savefig(destino / "barrido_eps.png", dpi=150); plt.show()

## 3 · Medir con personas que nunca se usaron

Cinco escenarios distintos con las personas de **prueba**. Se compara el `eps` elegido contra el del codo.

In [ ]:
def valor(x):
    return np.nan if x is None else float(x)

def medir(eps):
    salida = []
    for s in range(cfg["repeticiones"]):
        e = esc.construir(emb, etiquetas, personas_prueba, **cfg["escenario_prueba"],
                          ventana_dias=cfg["ventana_dias"], semilla=cfg["semilla"] + s)
        etq = cluster.agrupar(e.embeddings, eps, cfg["min_samples"])
        r = ev.evaluar_alertas(etq, e.personas, e.dias, cfg["min_apariciones"], cfg["min_dias"])
        r.update(cluster.resumen(e.embeddings, etq))
        r["semilla"] = cfg["semilla"] + s
        salida.append(r)
    return salida

def promedio(filas, clave):
    v = np.array([valor(f[clave]) for f in filas])
    return {"media": float(np.nanmean(v)), "desviacion": float(np.nanstd(v))} if not np.all(np.isnan(v)) else None

resultados = {}
for nombre, eps in (("eps_elegido", eps_elegido), ("eps_codo", round(eps_codo, 3))):
    filas_p = medir(eps)
    resultados[nombre] = {"eps": eps, "por_semilla": filas_p,
                          "precision": promedio(filas_p, "precision"), "recall": promedio(filas_p, "recall"),
                          "alertas": promedio(filas_p, "alertas"), "ruido": promedio(filas_p, "ruido")}
    print(nombre, eps, "| precisión", resultados[nombre]["precision"], "| recall", resultados[nombre]["recall"])
    for f in filas_p:
        print("   semilla", f["semilla"], {k: f[k] for k in ("alertas", "aciertos", "falsas_alarmas", "recurrentes_reales", "recurrentes_detectados")})

In [ ]:
from sklearn.decomposition import PCA

e = esc.construir(emb, etiquetas, personas_prueba, **cfg["escenario_prueba"],
                  ventana_dias=cfg["ventana_dias"], semilla=cfg["semilla"])
etq = cluster.agrupar(e.embeddings, eps_elegido, cfg["min_samples"])
reales = ev.personas_recurrentes(e.personas, e.dias, cfg["min_apariciones"], cfg["min_dias"])
alertan = set(cluster.clusters_recurrentes(etq, e.dias, cfg["min_apariciones"], cfg["min_dias"]))
p2 = PCA(n_components=2, random_state=cfg["semilla"]).fit_transform(e.embeddings)

fig, ejes = plt.subplots(1, 2, figsize=(12, 5))
ruido = etq == -1
ejes[0].scatter(*p2[ruido].T, c="lightgray", s=25, label="ruido")
for c in sorted(set(etq) - {-1}):
    m = etq == c
    ejes[0].scatter(*p2[m].T, s=45, label=f"clúster {c}" + (" (alerta)" if c in alertan else ""))
ejes[0].set_title(f"DBSCAN (eps = {eps_elegido})"); ejes[0].legend(fontsize=7)
es_rec = np.array([int(p) in reales for p in e.personas])
ejes[1].scatter(*p2[~es_rec].T, c="lightgray", s=25, label="no recurrentes (suelto, mismo día, dos días)")
for p in sorted(reales):
    m = e.personas == p
    ejes[1].scatter(*p2[m].T, s=45, label=f"persona {p}")
ejes[1].set_title("Verdad: quién es de verdad recurrente"); ejes[1].legend(fontsize=7)
for ax in ejes:
    ax.set_xticks([]); ax.set_yticks([])
fig.tight_layout(); fig.savefig(destino / "clusters_pca.png", dpi=150); plt.show()

## 4 · Guardar y descargar

Las métricas y las figuras se copian a `ml/experiments/<fecha>_dbscan_desconocidos/` del repo (sí se versionan).

In [ ]:
from google.colab import files

resultado = {
    "config": cfg, "eps_codo": eps_codo, "eps_elegido": eps_elegido,
    "fotos": int(len(emb)), "personas": int(len(np.unique(etiquetas))),
    "personas_ajuste": int(len(personas_ajuste)), "personas_prueba": int(len(personas_prueba)),
    "barrido_ajuste": filas, "prueba": resultados,
    "nota": "Simulación con fotos de LFW: las medidas son orientativas frente a una portería real.",
}
(destino / "metrics.json").write_text(
    json.dumps(resultado, indent=2, ensure_ascii=False, default=lambda o: o.item() if hasattr(o, "item") else str(o)),
    encoding="utf-8")
for nombre in ("metrics.json", "k_distance.png", "barrido_eps.png", "clusters_pca.png"):
    files.download(str(destino / nombre))